# Volcano

A comprehensive guide to Volcano for AI/ML workloads.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

Volcano is a **batch scheduler for Kubernetes** optimized for high-performance computing (HPC), big data, and AI/ML workloads.

### What is it?

- A Kubernetes scheduler and set of CRDs (e.g., **VolcanoJob**) for advanced batch scheduling.  
- Supports frameworks like **TensorFlow, PyTorch, Spark, Ray**, and more.  
- Provides features like gang scheduling, queueing, and topology-aware placement.

### Why use it?

Key benefits of using Volcano:

- **Advanced batch scheduling** beyond the default kube-scheduler.  
- **Framework integration** via VolcanoJob and other CRDs.  
- **Queueing, priority, and fairness** for compute-intensive workloads.

### When to use it?

Volcano is particularly useful when:

- You run large-scale **AI/ML or big data workloads** on Kubernetes.  
- You need **gang scheduling** or more advanced constraints than the default scheduler offers.  
- You want a unified batch scheduling layer for multiple frameworks.

## Key Features

### Core Capabilities of Volcano

| Feature | Description | Benefit |
|--------|-------------|---------|
| **VolcanoJob CRD** | Batch job abstraction tailored for AI/HPC workloads. | Richer semantics than plain Kubernetes Job. |
| **Gang scheduling** | Schedule a group of pods together or not at all. | Avoids partial allocation that hurts performance. |
| **Queues & priorities** | Manage multiple queues with different priorities and policies. | Fair sharing and SLA enforcement. |
| **Topology-aware scheduling** | Consider NUMA, GPU, and network topology. | Better performance for tightly coupled jobs. |
| **Framework support** | Integrations for TF, PyTorch, Ray, Spark, Flink, etc. | Unified scheduling across diverse workloads. |

## Architecture Overview

Volcano extends Kubernetes with its own scheduler and CRDs.

```text
+-----------------------------+
|   Users / Controllers       |
| (VolcanoJob, TFJob, etc.)   |
+---------------+-------------+
                |
                v
+-----------------------------+
| Volcano Controller &        |
| Scheduler                   |
+---------------+-------------+
                |
                v
+-----------------------------+
|   Kubernetes Nodes & Pods   |
+-----------------------------+
```

Volcano’s scheduler cooperates with Kubernetes control plane components to decide which nodes run which pods, considering queues, priorities, and gang scheduling requirements.

## Installation

Volcano is installed as:

- A scheduler component (often as an additional scheduler in the cluster).  
- CRDs (e.g., VolcanoJob) and controllers.

Installation is typically done via Helm charts or manifests, managed by platform teams. As an ML engineer, you mostly:

- Submit **VolcanoJob** resources.  
- Target Volcano queues and priorities configured by admins.

In [ ]:
# Volcano is installed on the cluster; you interact with its CRDs and queues.

print("Submit VolcanoJob resources once the Volcano scheduler and CRDs are installed.")

## Basic Usage

### Example: VolcanoJob (conceptual)

A minimal VolcanoJob for a distributed training job might look like this:

In [ ]:
# Example VolcanoJob manifest (YAML, conceptual)

volcano_job_yaml = """
apiVersion: batch.volcano.sh/v1alpha1
kind: Job
metadata:
  name: ml-training-job
spec:
  minAvailable: 4
  schedulerName: volcano
  queue: ml-queue
  tasks:
  - replicas: 4
    name: worker
    template:
      spec:
        restartPolicy: Never
        containers:
        - name: trainer
          image: your-registry/trainer:latest
          resources:
            requests:
              cpu: "4"
              memory: "16Gi"
              nvidia.com/gpu: 1
"""

print(volcano_job_yaml)

# Apply with: kubectl apply -f volcano-job.yaml

## Advanced Features

- **Gang scheduling via `minAvailable`**: Ensure all required pods for a job are scheduled together.  
- **Job lifecycle policies**: Control restart/cleanup behavior.  
- **Advanced queue configuration**: Multiple queues with different capacities and priorities.  
- **Resource fairness and partitioning**: Policies to balance usage across teams or projects.

In [ ]:
# Placeholder for advanced VolcanoJob examples

print("See the Volcano documentation for examples using advanced queues and gang scheduling.")

## Use Cases

- **Distributed AI/ML training** with strict gang scheduling requirements.  
- **Big data frameworks** like Spark and Flink running on Kubernetes.  
- **HPC-style workloads** that need advanced scheduling and topology-aware placement.  
- **Mixed workload clusters** where AI, analytics, and batch jobs share resources.

## Best Practices

1. **Understand scheduler semantics**  
   - Train users on gang scheduling, queues, and priorities.

2. **Use appropriate queues for workloads**  
   - Separate queues for production vs. research workloads if needed.

3. **Align Volcano with other controllers**  
   - Ensure training operators and other CRDs are configured to cooperate with Volcano.

4. **Test new policies gradually**  
   - Introduce new scheduling/queue policies in lower environments first.

## Common Pitfalls

1. **Misconfigured `minAvailable`**  
   - Symptom: Jobs remain pending because requirements exceed cluster capacity.  
   - Fix: Adjust `minAvailable` or scale cluster resources.

2. **Queue congestion**  
   - Symptom: Long wait times for certain queues.  
   - Fix: Re-balance workloads, adjust queue capacities or priorities.

3. **Interaction with default scheduler**  
   - Symptom: Pods not scheduled by Volcano when expected.  
   - Fix: Confirm `schedulerName` and cluster setup so Volcano handles relevant pods.

## Performance Optimization

- **Use topology-aware placement** for communication-heavy workloads.  
- **Align queue policies with hardware pools** (e.g., GPU vs CPU nodes).  
- **Monitor gang scheduling efficiency**: ensure that batch jobs are not starved due to overly strict constraints.

Performance tuning in Volcano is highly cluster- and workload-specific; combine scheduler metrics with application profiling.

In [ ]:
# Placeholder for metrics/benchmarking approach

print("Monitor Volcano queue and job metrics, along with pod/node utilization,\n"
      "to tune performance and policies.")

## Production Deployment

- **Cluster-wide scheduler**:  
  - Treat Volcano as a core part of the Kubernetes control plane for batch workloads.

- **Multi-tenant configuration**:  
  - Use queues and policies to separate teams, projects, or environments.  

- **Upgrade strategy**:  
  - Test new Volcano versions in staging before production rollout.

## Monitoring and Observability

- **Volcano metrics**:  
  - Use built-in metrics for queues, jobs, and scheduling behavior.  

- **Kubernetes monitoring**:  
  - Combine with Prometheus/Grafana for cluster and pod metrics.

- **Logging**:  
  - Collect Volcano scheduler and controller logs centrally for debugging.

## Troubleshooting

- **Jobs stuck in `Pending`**:  
  - Check queue configuration, `minAvailable`, and cluster capacity.

- **Unexpected scheduling decisions**:  
  - Inspect Volcano logs and events; review queue and policy configuration.

- **Compatibility issues**:  
  - Ensure your Kubernetes and Volcano versions are compatible; test controllers with simple jobs first.

## Comparison with Alternatives

| Aspect | Volcano | Kueue + Job/JobSet | Vendor GPU schedulers |
|--------|---------|--------------------|------------------------|
| Type | Batch scheduler with CRDs | Queueing + admission | Commercial platforms |
| Focus | AI/HPC/big data workloads | General batch queueing | Enterprise GPU utilization |
| Strengths | Gang scheduling, framework support | Native OSS queueing | Rich UI, advanced GPU sharing |

Choose Volcano when you:

- Need **advanced batch scheduling semantics** like gang scheduling for complex AI/HPC workloads on Kubernetes.

## Resources

- Volcano homepage: https://volcano.sh/en/  
- Documentation: https://volcano.sh/en/docs/  
- VolcanoJob docs: https://volcano.sh/en/docs/vcjob/

These resources provide installation instructions and job examples for different AI/ML frameworks.